« model_19 — SABİT NOKTALAR ÜZERİNDE ÖĞRENİLEN İLİŞKİLER · TÜRKÇE İLİŞKİ VERİSİ (data_tr) · TR DENEMELERİ »

**Soru: hikâye modelinde bulunan zaafların önerileri (R1, C1, E1, L1) hem ezberi hem çıkarımı ölçen bu dilde ne yapıyor?** Kullanıcı, 25 Eylül: *"bunların hepsini deneyeceğiz ama data_tr de deneyelim"* ve *"zincir öğretiyoruz ama sorularda öğretmediğimizi de soruyoruz modelin eğitim ve çıkarım performansını ölçüyoruz"*.
Veri ve karar kuralı koşudan önce `belge/onkayit/model_19.md` (TÜRKÇE ve TR denemeleri bölümleri); öneriler `belge/bulgu/model_19_analiz40k.md`.

| ne | değer |
|---|---|
| veri | model_16'nın `veri_16` hattı (izler birebir) · 1.608 varlık, 24 ilişki · 1.165.194 parça: bir bildirim ya da soru + cevabı · Drive `model_19/data_tr/` |
| token | doygun BPE, 1.483 · pencere `<eos> parça <eos>`, en uzun 28 · **yarım cümle yok** |
| model | D 1024 (512 + 512) · 256 hareket, aktif 8, 4 katman · attention 4 × 64 · defter · t_max 32 |
| eğitim | batch 1.024 · LR 0,002 · 20.000 adım (~17,6 epok) · 16.000 → 20.000 soğutma |
| ölçüm | her 500 adımda bölme başına 500 soru, birebir doğru cevap; bitişte bütün sınav; tam yedekte sınav sorularıyla decompose |

| sınav | derinlik | ne | soru |
|---|---|---|---|
| ezber_olgu | 1R | cevap eğitimde yazılı | 7.381 |
| ezber_zincir | 2R | zincir cevabıyla yazılı | 29.499 |
| ezber_3r | 3R | üç adımlı zincir cevabıyla yazılı | 3.000 |
| cikarim_gorulmemis | 2R | zincir hiç yazılmamış, tek adımlar yazılı | 2.747 |
| cikarim_yabanci | 2R | baş varlık hiç zincir başı olmamış | 3.195 |
| cikarim_3r | 3R | üç adımlı ve iki adımlı alt zincirler yazılmamış, üç olgu yazılı | 3.000 |

| koşu | tür | değişiklik |
|---|---|---|
| `TR_ARCH_BASE_S0` · `_S1` · `_S2` | model | zemin, tohum 0 · 1 · 2 |
| `TR_ARCH_R1` | model | defter gate'inde benzerlik girdisi yok |
| `TR_ARCH_C1` | model | ilk attention'ın sorgusuna ham zincir (induction) |
| `TR_ARCH_E1` | model | ikinci attention |
| `TR_TRAIN_L1` | eğitim | weight decay 0,1 (hareket + attention matrisleri) |
| `TR_ARCH_DESIGN` | model | TASARIM_19'un tamamı: E, R_PC, R_CC, mesafe eğilimi, ikinci attention |
| `TR_DATA_MORPH` | veri | kök ayrı token, ek ayrı token (model_16 `ek_16` → `tr_morph_19`: `▁kardeş i nin`); model TASARIM, 4 epok |
| `TR_DATA_MORPH_DOC` | veri | aynı token'lar, eğitim birimi BÜTÜN belge (sayfa, biyografi; model_16 gibi bölünmez); batch 128, 4 epok |
| `TR_ARCH_LOOP` | model | **Bul–Bak Döngüsü** (`looped_19.LoopedRelation`, TASARIM_19): üç ilişki girişte kaynak, tek blok en çok 4 geçiş, durum değişmeyince durur; veri bütün belge, batch 128, 4 epok. Mimari ve birim birden değişir: atfetme yok |
| `TR_ARCH_NORM` | model | referans `TR_DATA_MORPH` (aynı veri, aynı tarif) + Ö15: her katmanın ve attention'ın girdisi küreye (`norm_inputs`); 4 epok |

**Karar:** iki ana ölçü, EZBER (1R/2R/3R) ve ÇIKARIM (2R/2R/3R); deneme tek tohum, |deneme − zemin ortalaması| > zemin yayılımı ise **ADAY**; hüküm 3 tohumla (onay ister); biri artıp öteki düşerse TAKAS.

**Sıra:** `0 HAZIRLIK` → `1a` (zemin tohum 0: veri öğreniliyor mu) → `4 EĞRİ` · `6 SINAV` → `1h` TASARIM (kullanıcı: *"Kesinlikle tüm mimari kuralım"*) → `1i` VERİ (`DATA_NAME = 'data_tr_morph'`) → `1j` BELGE (`DATA_NAME = 'data_tr_morph_doc'`) → `1k` Ö15 (`DATA_NAME = 'data_tr_morph'`) → `1l` BUL–BAK (`DATA_NAME = 'data_tr_morph_doc'`) → `5 KARAR`
**Çekirdek düşerse:** `0 HAZIRLIK` → `2 SÜRDÜR`

**Dönen hücre YOK** (kural 8). **Koşular SIRAYLA** (torch.compile). **Koşuları kullanıcı başlatır (kural 0).**

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre, sonra "2 SURDUR".
import os, re, sys, subprocess

# Canli kosu varken moduller yeniden yuklenirse kosu listesi sifirlanir ve SURDUR ikinci bir kopya baslatir.
if 'train_19' in sys.modules:
    _live = [r.name for r in sys.modules['train_19'].RUNS.values() if r.alive]
    assert not _live, 'CEKIRDEK CANLI, kosu suruyor: %s -- HAZIRLIK gerekmiyor, 3 NABIZ' % _live

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_19'
# veri: data_tr (kelime basina token), data_tr_morph (kok ayri token, ek ayri token; birim cumle) ya da
# data_tr_morph_doc (ayni token'lar, birim BUTUN belge -- sayfa, biyografi; 26 Eylul)
DATA_NAME = 'data_tr_morph_doc'
DATA_DIR = ROOT + '/' + DATA_NAME
os.makedirs(ROOT, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
REPO = '/content/sekerai'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sekerahmet/sekerai.git', REPO], check=True)
SRC = REPO + '/deneme2/model_19'
CODE = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('kod   ' + CODE)
assert all(os.path.exists(SRC + '/' + f) for f in ('data_tr_19.py', 'looped_19.py')), \
    'depoda data_tr_19 ya da looped_19 YOK -- yerelde git push gerekli (train_19 looped_19\'u ice aktarir)'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in ('model_19', 'train_19', 'data_stories_19', 'decompose_19', 'diagnose_19', 'data_tr_19', 'tr_graph_19',
           'tr_text_19', 'tr_corpus_19', 'tr_splits_19', 'tr_chars_19', 'tr_morph_19', 'looped_19'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import model_19
import looped_19
import train_19 as train
import data_tr_19 as TR

# Kural 9: veri Drive'dan; token ve tokenizer izi yeniden hesaplanip karsilastirilir.  Colab veri URETMEZ.
D = TR.load(DATA_DIR)
W, M = TR.make_windows(D)
TRAIN = (W, M, M)                      # kayip butun token'larda
VOCAB, N = D['vocab'], len(D['vocab'])
TOK = TR.tokenizer_from(D['tokenizer'])
METRIC = TR.make_metric(D, (W, M), device='cuda', limit=2000, exam_limit=500)
EZBER = ('ezber_olgu', 'ezber_zincir', 'ezber_3r')
CIKARIM = ('cikarim_gorulmemis', 'cikarim_yabanci', 'cikarim_3r')

# Onkayit (belge/onkayit/model_19.md, TR denemeleri): batch 1024, 20.000 adim, 16.000'den sogutma, olcum/agirlik 500,
# tam yedek 2.000.  Kullanici, 25 Eylul: "D 1024, hikâyedekiyle aynı".  t_max 32: en uzun pencere 28.
COMMON = dict(batch=1024, steps=20000, eval_every=500, save_every=2000, weights_every=500, decay_start=16000)
MODEL = dict(d_order=512, d_content=512, vectors=256, active=8, layers=4, attn_heads=4, attn_dim=64, t_max=32, rank=256)
BASE = dict(embed=False, readout=False, chain_sim=False, distance=False, attn_after=(0,), induction_query=False,
            gate_sim=True)                                                                        # zemin
MODEL_BASE = dict(MODEL, **BASE)

# IKI TUR DENEME AYRI IZLENIR (kullanici, 25 Eylul).  Ikisini birden degistiren kosu BASLAMAZ.
REF_RECIPE = dict(lr=0.002, warmup=0, decay_floor=0.1, weight_decay=0.0)
TARIF = dict(REF_RECIPE)
TRAIN_TRIALS = {'TR_TRAIN_L1': dict(weight_decay=0.1)}          # L1: AdamW, yalniz hareket ve attention matrisleri
ARCH_TRIALS = {'TR_ARCH_BASE_S%d' % s: (s, MODEL_BASE) for s in (0, 1, 2)}
for part, kw in (('R1', dict(gate_sim=False)), ('C1', dict(induction_query=True)), ('E1', dict(attn_after=(0, 2)))):
    ARCH_TRIALS['TR_ARCH_' + part] = (0, dict(MODEL_BASE, **kw))
# TASARIM_19'un tamami birden (kullanici, 25 Eylul: "Kesinlikle tüm mimari kuralım")
DESIGN = dict(embed=True, readout=True, chain_sim=True, distance=True, attn_after=(0, 2))
ARCH_TRIALS['TR_ARCH_DESIGN'] = (0, dict(MODEL_BASE, **DESIGN))
# veri denemesi: model TASARIM, tarif TARIF, yalniz veri (kok neden: OLCULENLER §1l, 26 Eylul)
# ad -> (veri, model farki, tarif farki).  t_max bellek korumasi (konum siniri degil): en uzun pencere 35 / 368.
# Belgede batch 128: 128 x ~121 token = cumle kosusunun 1024 x ~15'i, adim basina ayni gercek token.
DATA_TRIALS = {'TR_DATA_MORPH': ('data_tr_morph', dict(t_max=40), dict()),
               'TR_DATA_MORPH_DOC': ('data_tr_morph_doc', dict(t_max=384), dict(batch=128))}

# model denemesi kok+ek verisinde: referans TR_DATA_MORPH (TASARIM, t_max 40), tek degisiklik model.  ad -> (veri, model farki)
MORPH_BASE = dict(MODEL_BASE, **DESIGN, t_max=40)
MORPH_ARCH_TRIALS = {'TR_ARCH_NORM': ('data_tr_morph', dict(norm_inputs=True))}    # Oe15: girdi kureye (26 Eylul)

# Bul-Bak Dongusu (TASARIM_19, 26 Eylul): yeni mimari, veri butun belge.  Mimari ve birim birden degisir -- atfetme
# yapilamaz (CLAUDE.md: "iki dugme birden -> ATFETME yapilamaz diye YAZILIR, kol kosulur").  ad -> (veri, model, tarif farki)
# step_norm ACIKCA (varsayilan degisirse kosu kaymasin): 'input' CPU'da coktu (OLCULENLER), 'bounded' oneri --
# kullanici karari bekliyor; koşu zaten onayla baslar.
NEW_ARCH_TRIALS = {'TR_ARCH_LOOP': ('data_tr_morph_doc', dict(arch='looped_relation', t_max=512, step_norm='bounded'),
                                    dict(batch=128))}

# ad -> (tur, egitim ayari, model ayari)
CONFIGS = {**{n: ('egitim', dict(TARIF, **tr, seed=0), MODEL_BASE) for n, tr in TRAIN_TRIALS.items()},
           **{n: ('model', dict(TARIF, seed=s), mk) for n, (s, mk) in ARCH_TRIALS.items()},
           **{n: ('veri', dict(TARIF, seed=0, **rc), dict(MODEL_BASE, **DESIGN, **mc))
              for n, (_, mc, rc) in DATA_TRIALS.items()},
           **{n: ('model', dict(TARIF, seed=0), dict(MORPH_BASE, **mc)) for n, (_, mc) in MORPH_ARCH_TRIALS.items()},
           **{n: ('model', dict(TARIF, seed=0, **rc), dict(mc)) for n, (_, mc, rc) in NEW_ARCH_TRIALS.items()}}
EXTRA = dict(data='%s token %s bpe %s' % (DATA_NAME, D['ids_fp'], D['tokenizer_fp']), fingerprint=D['parts_fp'],
             exam=D['exam_fp'], T=W.shape[1], code=CODE, onkayit='belge/onkayit/model_19.md')
GPU_MARGIN = 1.2            # tahmin fazla: model_18'de tahmin 7,6 GB, nvidia-smi 5,2 GB (REL07, 24 Eylul)


def trial(name):
    '''-> (tur, referansa gore degisen).  Ikisini birden degistiren deneme DURUR.'''
    kind, tr, mk = CONFIGS[name]
    if kind == 'veri':
        return kind, dict({'veri': DATA_TRIALS[name][0]}, **DATA_TRIALS[name][1], **DATA_TRIALS[name][2])
    if name in NEW_ARCH_TRIALS:                         # yeni mimari: degisen her sey acikca
        data, mc, rc = NEW_ARCH_TRIALS[name]
        return kind, dict(mc, veri=data, **rc)
    recipe = {k: tr.get(k, REF_RECIPE[k]) for k in REF_RECIPE}
    ref = MORPH_BASE if name in MORPH_ARCH_TRIALS else MODEL_BASE
    model = {k: v for k, v in mk.items() if ref.get(k) != v}
    if kind == 'egitim':
        assert not model, name + ': egitim denemesi modeli degistiremez -- ' + str(model)
        return kind, {k: v for k, v in recipe.items() if REF_RECIPE[k] != v}
    assert recipe == {k: TARIF.get(k, REF_RECIPE[k]) for k in REF_RECIPE}, name + ': model denemesi tarifi degistiremez'
    return kind, model


def memory_gb(d_order, d_content, vectors, active, layers, attn_heads, attn_dim, attn_after=(0,), batch=None, **_):
    '''Geri yayilim icin saklanan, TAHMIN (TR'de olculmedi): hikayedekiyle ayni kalemler, T pencere boyu.'''
    B, T, d = batch or COMMON['batch'], W.shape[1], d_order + d_content
    attn = len(attn_after) * (4 * attn_heads * attn_dim + attn_heads * T)
    return B * T * (layers * (vectors + active * d + 3 * d) + 7 * N + 7 * T + 6 * d_content + attn) * 4 / 1e9


def launch(name, resume=None, steps=None, **plan):
    '''CONFIGS[name] ile arka planda baslatir (kural 8); GPU kapisi hucrenin kendisinde.'''
    kind, tr, mk = CONFIGS[name]
    kind, changed = trial(name)
    want = (DATA_TRIALS.get(name) or MORPH_ARCH_TRIALS.get(name) or NEW_ARCH_TRIALS.get(name) or ('data_tr',))[0]
    assert want == DATA_NAME, '%s %s verisi ister, yuklu olan %s -- HAZIRLIK\'ta DATA_NAME' % (name, want, DATA_NAME)
    kw = dict(COMMON, **tr)
    kw.update(plan)
    if steps is not None:
        kw['steps'] = steps
    return train.start(name, TRAIN, N, metric=METRIC, device='cuda', root=ROOT, vocab=VOCAB, resume=resume,
                       extra=dict(EXTRA, trial=kind, changed=changed), **kw, **mk)


def epochs(name, k=4):
    '''k epok kac adim -- denemenin kendi batch'iyle (kullanici, 26 Eylul: "1 ile 4 epoch arası istenirse uzar").'''
    return round(k * len(W) / CONFIGS[name][1].get('batch', COMMON['batch']))


def memory_gb_looped(batch=None, d=looped_19.D, vectors=looped_19.VECTORS, heads=looped_19.HEADS,
                     head_dim=looped_19.HEAD_DIM, k_max=looped_19.K_MAX, step_norm=looped_19.STEP_NORM, **_):
    '''LoopedRelation, geri yayilim icin saklanan, TAHMIN (olculmedi): gecis basina attention (2 H T), q k v cikti (4 H dh),
    iki bak katmani (uzaklik, agirlik, denge olasiligi 3 V + d), kureye inisler (6 d); kaynaklar ve cikis bir kez.'''
    B, T = batch or COMMON['batch'], W.shape[1]
    per_pass = 2 * heads * T + 4 * heads * head_dim + 2 * (3 * vectors + d) + 6 * d
    if step_norm == 'bounded':                          # parca ve toplam da kureye: denetim olcumu +9 d / gecis
        per_pass += 9 * d
    return B * T * (k_max * per_pass + T + 6 * d + 3 * N) * 4 / 1e9


def need_gb(name):
    '''GPU kapisi icin: denemenin kendi batch'iyle bellek tahmini (mimariye gore).'''
    mk, batch = CONFIGS[name][2], CONFIGS[name][1].get('batch')
    return memory_gb_looped(batch=batch, **mk) if mk.get('arch') == 'looped_relation' else memory_gb(**mk, batch=batch)


def latest_model(name):
    '''Diskteki EN YENI agirlik (w ya da t) -- kosu surerken de, ayri bir kopya.'''
    d = ROOT + '/' + name
    f = max((int(re.findall('[0-9]+', x)[0]), x) for x in os.listdir(d) if re.match('[tw][0-9]+[.]pt$', x))[1]
    k = torch.load(d + '/' + f, weights_only=False, map_location='cuda')
    return looped_19.model_from_package(k).cuda().eval(), k


n = len(W)
print('veri %s (birim %s)   ' % (DATA_NAME, D.get('unit', 'part')) + 'sozluk %d token   parca %s   pencere T=%d   1 epok = %s adim   %s adim = %.1f epok'
      % (N, f'{n:,}', W.shape[1], f"{n // COMMON['batch']:,}", f"{COMMON['steps']:,}", COMMON['steps'] * COMMON['batch'] / n))
print('sinav: ' + '   '.join('%s %d' % (k, len(D['exam'][k])) for k in EZBER + CIKARIM))
for kind_title, kind_key in (('MODEL DENEMELERI (tarif = TARIF)', 'model'), ('EGITIM DENEMELERI (model = zemin)', 'egitim'),
                             ('VERI DENEMELERI (model = TASARIM, tarif = TARIF)', 'veri')):
    names = [x for x in CONFIGS if CONFIGS[x][0] == kind_key]
    print('\n' + kind_title)
    for name in names:
        _mk = dict(CONFIGS[name][2])
        _m = train.ARCHS[_mk.pop('arch', 'point_relation')](N, **_mk)
        print('  %-18s tohum %d   degisen %-28s parametre %s   saklanan ~%.1f GB'
              % (name, CONFIGS[name][1]['seed'], trial(name)[1], f'{sum(p.numel() for p in _m.parameters()):,}',
                 need_gb(name)))
del _m
print('\nORNEK PENCERELER')
for i in range(3):
    print('  ' + TR.detok(TOK, W[i][1:int(M[i].sum()) - 1].tolist()))
print('ORNEK SORULAR')
for k in EZBER + CIKARIM:
    q, a = D['exam'][k][0]
    print('  %-20s %s  ->  %s' % (k, q, a))

In [ ]:
# 1a KOSU -- zemin, tohum 0 (ILK: veri ogreniliyor mu)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_BASE_S0'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1b KOSU -- R1: gate'te benzerlik girdisi yok  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_R1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1c KOSU -- C1: ilk attention'in sorgusuna ham zincir  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_C1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1d KOSU -- E1: ikinci attention  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_E1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1e KOSU -- L1 (EGITIM): weight decay 0,1  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_TRAIN_L1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1f KOSU -- zemin, tohum 1  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_BASE_S1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1g KOSU -- zemin, tohum 2  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_BASE_S2'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1h KOSU -- TASARIM: E, R_PC, R_CC, mesafe, ikinci attention  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_DESIGN'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1i KOSU -- VERI: kok ayri token, ek ayri token (birim cumle), model TASARIM, 4 epok  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_DATA_MORPH'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * need_gb(NAME))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME, steps=epochs(NAME), decay_start=int(0.8 * epochs(NAME))))   # 4 epok, son %20 sogutma

In [ ]:
# 1j KOSU -- VERI: kok+ek token'lari, birim BUTUN belge (sayfa, biyografi), model TASARIM, 4 epok  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_DATA_MORPH_DOC'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * need_gb(NAME))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME, steps=epochs(NAME), decay_start=int(0.8 * epochs(NAME))))   # 4 epok, son %20 sogutma

In [ ]:
# 1k KOSU -- MODEL (Oe15): TR_DATA_MORPH + her katmanin ve attention'in girdisi kureye, 4 epok  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_NORM'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * need_gb(NAME))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME, steps=epochs(NAME), decay_start=int(0.8 * epochs(NAME))))   # 4 epok, son %20 sogutma

In [ ]:
# 1l KOSU -- MIMARI: LoopedRelation, Bul-Bak Dongusu (looped_19), veri butun belge, batch 128, 4 epok  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'TR_ARCH_LOOP'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * need_gb(NAME))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME, steps=epochs(NAME), decay_start=int(0.8 * epochs(NAME))))   # 4 epok, son %20 sogutma

In [ ]:
# 2 SURDUR  |  GPU  |  tekrar: GUVENLI
# Cekirdek dustuyse: once "0 HAZIRLIK", sonra BU hucre.  NAME'i sec.
# Kural 1: uzatma SURDURMEDIR -- uzatmak icin STEPS'i buyut, bu hucreyi calistir.
import gc, os, re, torch
NAME = 'TR_ARCH_LOOP'           # CONFIGS'teki adlardan biri
STEPS = None                    # None: paketin kendi plani (decay_end); UZATMAK icin sayi ver (kural 1)
PLAN = None                     # None: paketin sogutmasi; uzatmada yeni plan, ornek dict(decay_start=32000)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * need_gb(NAME))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

_d = ROOT + '/' + NAME
_n = sorted((int(re.findall('[0-9]+', f)[0]), f) for f in os.listdir(_d) if re.match('t[0-9]+[.]pt$', f))
assert _n, 'surdurme paketi YOK -- ' + _d
RESUME_FROM = _d + '/' + _n[-1][1]
_pk = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
STEPS = STEPS or _pk.get('decay_end') or COMMON['steps']          # sessizce 20.000'e kaymasin
PLAN = PLAN if PLAN is not None else ({} if _pk.get('decay_start') is None else dict(decay_start=_pk['decay_start']))
del _pk
print('son nokta  %s   adim %s   hedef %s' % (RESUME_FROM, f'{_n[-1][0]:,}', f'{STEPS:,}'))
assert _n[-1][0] < STEPS, 'zaten hedefe varmis -- uzatmak icin STEPS buyut'
print(launch(NAME, resume=RESUME_FROM, steps=STEPS, **PLAN))

In [ ]:
# 3 NABIZ  |  CPU  |  tekrar: GUVENLI
# DONMEZ, hemen doner.  HICBIR SEY KOSTURMAZ (kural 8) -- yalniz gunlugu ve GPU bellek tepesini basar.
import gc
import torch
_runs = [o for o in gc.get_objects() if type(o).__name__ == 'Run']
for _r in sorted(_runs, key=lambda r: not r.alive):
    print('%s: %s   gunluk %d satir%s' % (_r.name, 'CANLI' if _r.alive else 'bitti', len(_r.log),
                                          '   DURDUR istendi' if _r.stop_requested else ''))
    for _s in (_r.log[-30:] if _r.alive else _r.log[-1:]):
        print(_s)
if torch.cuda.is_available():
    print('\nGPU bellek tepesi (bu cekirdekte) %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))

In [ ]:
# 4 EGRI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar, hicbir sey kosturmaz
# Gunlukten: her 500 adimda bolme basina 500 soru.  Ust: EZBER ve CIKARIM ortalamasi.  Alt: HER ADIMIN kaybi.
import os
import torch
import matplotlib.pyplot as plt


def columns(name):
    '''gunluk.txt -> {adim: {sutun: deger}}; sutun adlari gunlugun baslik satirindan.'''
    path, head, out = ROOT + '/' + name + '/gunluk.txt', None, {}
    if not os.path.exists(path):
        return out
    for s in open(path, encoding='utf-8'):
        p = s.split()
        if len(p) > 3 and p[1] == 'step' and 'heldout' in p:
            head = p[1:]
        elif head and len(p) > 3 and p[1].isdigit():
            try:
                out[int(p[1])] = {h: float(x) for h, x in zip(head, p[1:])}
            except ValueError:
                pass
    return out


fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for name in CONFIGS:
    r = columns(name)
    if not r:
        continue
    x = sorted(r)
    ez = [sum(r[i].get(k, float('nan')) for k in EZBER) / 3 for i in x]
    ci = [sum(r[i].get(k, float('nan')) for k in CIKARIM) / 3 for i in x]
    a1.plot(x, ez, label=name + ' ezber')
    a1.plot(x, ci, ':', label=name + ' cikarim')
    last = r[x[-1]]
    print('%-18s adim %6s   ezber %.3f (%s)   cikarim %.3f (%s)' % (
        name, f'{x[-1]:,}', ez[-1], ' '.join('%.2f' % last.get(k, float('nan')) for k in EZBER),
        ci[-1], ' '.join('%.2f' % last.get(k, float('nan')) for k in CIKARIM)))
    k = train.RUNS[name].result.get('step_losses') if name in train.RUNS else None
    if k is not None:
        k = k[~k.isnan()]
        a2.plot(k.numpy(), lw=0.4, label=name)
a1.set_ylabel('birebir dogru cevap')
a1.legend(fontsize=7, ncol=2)
a1.grid(alpha=0.3)
a2.set_yscale('log')
a2.set_ylabel('her adimin kaybi')
a2.set_xlabel('adim')
a2.grid(alpha=0.3)
plt.show()

In [ ]:
# 5 KARAR  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar; kural belge/onkayit/model_19.md (TR denemeleri), aynen
# Bitisteki TAM sinav.  EZBER = ezber_olgu/zincir/3r, CIKARIM = cikarim_gorulmemis/yabanci/3r (agirliksiz ortalama).
# Zemin 3 tohum -> m, yayilim y.  Deneme tek tohum: |deneme - m| > y ise ADAY; biri artip oteki duserse TAKAS.
import math, os, torch


def final(name):
    '''-> {EZBER, CIKARIM, bolmeler} bitisteki tam olcumden; bitmediyse None.'''
    p = ROOT + '/model_' + name + '.pt'
    if not os.path.exists(p):
        return None
    k = torch.load(p, weights_only=False, map_location='cpu')
    if not k.get('done'):
        return None
    g = k['heldout_diag']
    k['step_losses'] = k['step_losses'][~k['step_losses'].isnan()]
    div = (not math.isfinite(float(k['step_losses'][-1]))) or \
        float(k['step_losses'][-1000:].mean()) > float(k['step_losses'][:500].mean())
    return dict(EZBER=sum(g[x] for x in EZBER) / 3, CIKARIM=sum(g[x] for x in CIKARIM) / 3, diag=g, div=div)


base = {n: final(n) for n in ARCH_TRIALS if n.startswith('TR_ARCH_BASE_')}
for n, r in base.items():
    print('%-18s %s' % (n, 'ezber %.4f  cikarim %.4f%s' % (r['EZBER'], r['CIKARIM'], '  IRAKSADI' if r['div'] else '')
                        if r else 'BITMEDI'))
done = [r for r in base.values() if r]
stat = {}
if len(done) == 3:
    for key in ('EZBER', 'CIKARIM'):
        v = [r[key] for r in done]
        stat[key] = (sum(v) / 3, max(v) - min(v))
        print('zemin %-8s m %.4f   yayilim y %.4f' % (key, *stat[key]))
else:
    print('zemin %d / 3 tohum -- hukum icin uc tohum gerekir' % len(done))
print()
for n in list(ARCH_TRIALS) + list(TRAIN_TRIALS):
    if n.startswith('TR_ARCH_BASE_'):
        continue
    r = final(n)
    if not r:
        print('%-18s BITMEDI' % n)
        continue
    line = '%-18s ezber %.4f  cikarim %.4f' % (n, r['EZBER'], r['CIKARIM'])
    if r['div']:
        line += '   IRAKSADI -- ARIZA: tohum eklenmez, decompose ile incelenir'
    elif stat:
        sign = {}
        for key in ('EZBER', 'CIKARIM'):
            m, y = stat[key]
            d = r[key] - m
            sign[key] = (d > 0) - (d < 0) if abs(d) > y else 0
            line += '   %s %+.2f puan %s' % (key.lower(), 100 * d, 'ADAY(%s)' % '+-'[d < 0] if sign[key] else '-')
        if sign['EZBER'] * sign['CIKARIM'] < 0:
            line += '   TAKAS'
        if any(sign.values()):
            line += '   -> tohum 1 ve 2 ONAY ister'
    print(line)
    print('      ' + '  '.join('%s %.3f' % (k, r['diag'][k]) for k in EZBER + CIKARIM))

In [ ]:
# 6 SINAV  |  GPU  |  ARKA PLAN: ilk calistirma baslatir, sonrakiler okur  |  tekrar: GUVENLI
# SAYI degil METIN (kural 12): her bolmeden sabit tohumla 6 soru, modelin cevabi ve dogrusu.  Diskteki EN YENI agirlik.
import random, threading
NAME = 'TR_ARCH_BASE_S0'
PER_SPLIT = 6
REDO = False                    # True: bitmis isi diskteki yeni agirlikla bastan yap

JOBS = globals().setdefault('JOBS', {})
_run_name = 'SINAV ' + NAME
if _run_name not in JOBS or (REDO and not JOBS[_run_name].alive):
    # --- GPU KAPISI (CLAUDE.md kural 2)
    assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'

    def _exam_job(r, name=NAME):
        try:
            m, k = latest_model(name)
            r.note('%s   adim %s' % (name, f"{k['step']:,}"))
            for split in EZBER + CIKARIM:
                pairs = random.Random(7).sample(D['exam'][split], PER_SPLIT)
                said = TR.ask(m, TOK, [q for q, _ in pairs], device='cuda')
                ok = sum(TR.correct(s, a) for s, (_, a) in zip(said, pairs))
                r.note('-- %s  (%d / %d)' % (split, ok, PER_SPLIT))
                for s, (q, a) in zip(said, pairs):
                    r.note('   %s %s\n       model: %s\n       dogru: %s' % ('+' if TR.correct(s, a) else 'x', q, s, a))
        except Exception as h:
            r.note('HATA  %r' % h)

    JOBS[_run_name] = train.Run(_run_name)
    JOBS[_run_name].thread = threading.Thread(target=_exam_job, args=(JOBS[_run_name],), daemon=True)
    JOBS[_run_name].thread.start()
_r = JOBS[_run_name]
print('%s: %s' % (_run_name, 'SORUYOR -- hucreyi tekrar calistir' if _r.alive else 'bitti'))
for _s in _r.log:
    print(_s.split('] ', 1)[1])

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki adimda CIKMADAN ONCE Drive'a kaydeder.
train.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for name in CONFIGS:
    _d = ROOT + '/' + name
    if not os.path.isdir(_d):
        print('%-18s henuz kayit yok' % name)
        continue
    _f = sorted(os.listdir(_d))
    _b = sum(os.path.getsize(_d + '/' + f) for f in _f if os.path.isfile(_d + '/' + f))
    print('%-18s %3d yedek   %.3f GB   %s' % (name, sum(f.endswith('.pt') for f in _f), _b / 1e9, _d))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi